In [1]:
# ==============================================================================
# STEP 1: ENVIRONMENT VERIFICATION & HARDWARE AUDIT
# ==============================================================================
import os, sys

# Must happen BEFORE torch is imported: use a single GPU (prevents accidental cross-GPU sharding)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if "torch" in sys.modules:
    print("WARNING: torch was imported before this cell. If restarting, run from the top.")

import importlib.util, subprocess, random, json, time, re, shutil, math, gc, inspect, base64
from typing import Dict, List, Any, Optional

REQUIRED = {
    "transformers": "transformers", "peft": "peft", "bitsandbytes": "bitsandbytes",
    "accelerate": "accelerate", "datasets": "datasets", "sklearn": "scikit-learn",
    "seaborn": "seaborn", "matplotlib": "matplotlib", "pandas": "pandas", "safetensors": "safetensors",
}
missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f"Installing missing packages: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)

import torch, transformers, peft, accelerate, sklearn
import numpy as np, pandas as pd
import bitsandbytes as bnb

print("=" * 80)
print("DEPENDENCY & HARDWARE AUDIT")
print("=" * 80)
print(f"PyTorch:        {torch.__version__}")
print(f"Transformers:   {transformers.__version__}")
print(f"PEFT:           {peft.__version__}")
print(f"Accelerate:     {accelerate.__version__}")
print(f"BitsAndBytes:   {bnb.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. In Kaggle: Notebook settings -> Accelerator -> 'GPU T4 x2', "
        "then restart the session. 4-bit QLoRA cannot run on CPU."
    )

props = torch.cuda.get_device_properties(0)
print(f"Visible GPUs:   {torch.cuda.device_count()} (restricted to GPU 0 on purpose)")
print(f"GPU 0:          {props.name} | {props.total_memory / 1024**3:.2f} GB | compute capability {props.major}.{props.minor}")
print("=" * 80)

Installing missing packages: ['bitsandbytes']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 38.8 MB/s eta 0:00:00
DEPENDENCY & HARDWARE AUDIT
PyTorch:        2.11.0+cu128
Transformers:   5.16.1
PEFT:           0.20.0
Accelerate:     1.14.0
BitsAndBytes:   0.50.2
CUDA available: True
Visible GPUs:   1 (restricted to GPU 0 on purpose)
GPU 0:          Tesla T4 | 14.56 GB | compute capability 7.5


In [2]:
# ==============================================================================
# STEP 2: MASTER CONFIGURATION & STRICT 5K DATASET PATH RESOLVER
# ==============================================================================
from transformers import set_seed

RANDOM_SEED = 42
set_seed(RANDOM_SEED)

# 1. Hardware / precision (T4 = compute capability 7.5 -> FP16; Ampere/Ada >= 8 -> BF16)
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
CC_MAJOR = torch.cuda.get_device_capability(0)[0]
USE_BF16 = (CC_MAJOR >= 8) and torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"Device: {GPU_NAME} | VRAM: {VRAM_GB:.2f} GB | Precision: {'BF16' if USE_BF16 else 'FP16'}")

# 2. Dataset discovery (Strictly take only the 5k dataset: llm_instruction_5k.jsonl)
TARGET_DATASET_FILENAME = "llm_instruction_5k.jsonl"

CANDIDATE_DIRS = [
    r"C:\Users\USER\OneDrive\Pictures\raita_all\paper collection\archive\processed_data",
    "./archive/processed_data",
    "../archive/processed_data",
    "processed_data",
    "/kaggle/input/datasets/siamabir/rhetorical-roles-processed-data/processed_data",
    "/kaggle/input/datasets/tasnimsaimaraita/processed-dataset/processed_data",
    "/kaggle/input/datasets/tasnimsaimaraita/processed-dataset",
    "/kaggle/input/rhetorical-roles-processed-data/processed_data",
    "/kaggle/input/datasets/siamabir/rhetorical-roles-processed-data",
    "/kaggle/input/rhetorical-roles-processed-data",
    "/kaggle/input/archive/processed_data",
    "/kaggle/input",
    "."
]

DATA_PATH = None
for d in CANDIDATE_DIRS:
    if d and os.path.exists(d):
        candidate_file = os.path.join(d, TARGET_DATASET_FILENAME)
        if os.path.isfile(candidate_file):
            DATA_PATH = candidate_file
            break
        for root, _, files in os.walk(d):
            if TARGET_DATASET_FILENAME in files:
                DATA_PATH = os.path.join(root, TARGET_DATASET_FILENAME)
                break
        if DATA_PATH:
            break

if not DATA_PATH or not os.path.isfile(DATA_PATH):
    raise FileNotFoundError(
        f"CRITICAL ERROR: Could not find '{TARGET_DATASET_FILENAME}'. "
        f"Verified candidate directories: {CANDIDATE_DIRS}"
    )

print(f"Using strict 5k instruction dataset: {DATA_PATH} ({os.path.getsize(DATA_PATH) / 1024**2:.1f} MB)")

# 3. Model parameters
MODEL_ID = "BioMistral/BioMistral-7B"
SHORT_NAME = "BioMistral-7B"
IS_GATED = False

TARGET_CLASSES = ["BACKGROUND", "OBJECTIVE", "METHODS", "RESULTS", "CONCLUSION"]
NUM_CLASSES = len(TARGET_CLASSES)
UNPARSEABLE_ID = NUM_CLASSES
LABEL2ID = {lbl: i for i, lbl in enumerate(TARGET_CLASSES)}
ID2LABEL = {i: lbl for lbl, i in LABEL2ID.items()}

# LoRA Configuration
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Training (Single GPU safe batching: 4 x 4 = 16)
PER_DEVICE_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
EFFECTIVE_BATCH = PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
MAX_SEQ_LENGTH = 512
LEARNING_RATE = 2e-4
NUM_TRAIN_EPOCHS = 1
WARMUP_RATIO = 0.03
OPTIMIZER_NAME = "paged_adamw_8bit"

# Data size / time budget controls
MAX_TRAIN_SAMPLES = None
AUTO_FIT_TO_BUDGET = True
MAX_TRAIN_HOURS = 8.5
VAL_EVAL_SAMPLES = 400
TEST_EVAL_SAMPLES = 1000
EVAL_BATCH_SIZE = 8
MAX_NEW_TOKENS = 10
LOG_STEPS = 25
SAVE_STEPS = 250

WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "./output"
MODELS_OUTPUT_DIR = os.path.join(WORKING_DIR, "models", "llm_qlora")
output_run_dir = os.path.join(MODELS_OUTPUT_DIR, SHORT_NAME)
checkpoint_dir = os.path.join(WORKING_DIR, "checkpoints", SHORT_NAME)
os.makedirs(output_run_dir, exist_ok=True)
os.makedirs(checkpoint_dir, exist_ok=True)

print(f"Active Model: {SHORT_NAME} ({MODEL_ID})")


Device: Tesla T4 | VRAM: 14.56 GB | Precision: FP16
Using strict 5k instruction dataset: /kaggle/input/datasets/siamabir/rhetorical-roles-processed-data/processed_data/llm_instruction_5k.jsonl (77.7 MB)
Active Model: BioMistral-7B (BioMistral/BioMistral-7B)


In [3]:
# ==============================================================================
# STEP 3: HUGGING FACE AUTHENTICATION (FOR GATED MODELS / HIGHER RATE LIMITS)
# ==============================================================================
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        hf_token = None

if hf_token:
    from huggingface_hub import login
    login(token=hf_token)
    print("Logged in to Hugging Face Hub.")
else:
    print(f"Proceeding anonymously for public model: {SHORT_NAME}")

Proceeding anonymously for public model: BioMistral-7B


In [4]:
# ==============================================================================
# STEP 4: LOAD & PARTITION THE 5K CHATML DATASET
# ==============================================================================
from datasets import Dataset

print(f"Loading records from: {DATA_PATH} ...")
raw_records = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            raw_records.append(json.loads(line))
print(f"Total records in 5k dataset: {len(raw_records):,}")

def get_label(rec) -> str:
    msgs = rec["messages"]
    asst = [m["content"].strip() for m in msgs if m.get("role") == "assistant"]
    return asst[-1] if asst else msgs[-1]["content"].strip()

# Verify all records have valid labels
valid_records, n_bad = [], 0
for r in raw_records:
    if get_label(r) in LABEL2ID:
        valid_records.append(r)
    else:
        n_bad += 1
if n_bad > 0:
    print(f"Dropped {n_bad} records with unknown label.")

train_records     = [r for r in valid_records if r.get("split") == "train"]
val_records       = [r for r in valid_records if r.get("split") == "val"]
test_records_full = [r for r in valid_records if r.get("split") == "test"]
print(f"Partitions: train={len(train_records):,} | val={len(val_records):,} | test={len(test_records_full):,}")

rng = random.Random(RANDOM_SEED)
rng.shuffle(train_records)

# Validation subset for rapid training loss evaluation
if VAL_EVAL_SAMPLES is not None and VAL_EVAL_SAMPLES < len(val_records):
    val_records = rng.sample(val_records, VAL_EVAL_SAMPLES)

# Test subset: intact whole abstracts (preserves rhetorical transition consistency)
def select_test_by_paper(records, n, rng):
    if n is None or n >= len(records):
        return records
    groups = {}
    for r in records:
        groups.setdefault(r.get("paper_id", "unknown"), []).append(r)
    pids = list(groups.keys())
    rng.shuffle(pids)
    out = []
    for pid in pids:
        out.extend(groups[pid])
        if len(out) >= n:
            break
    return out

test_records = select_test_by_paper(test_records_full, TEST_EVAL_SAMPLES, rng)

ds_train = Dataset.from_list(train_records)
ds_val   = Dataset.from_list(val_records)
ds_test  = Dataset.from_list(test_records)
print(f"Active datasets: train={len(ds_train):,} | val={len(ds_val):,} | test={len(ds_test):,} "
      f"(intact test abstracts: {len({r.get('paper_id') for r in test_records})})")

Loading records from: /kaggle/input/datasets/siamabir/rhetorical-roles-processed-data/processed_data/llm_instruction_5k.jsonl ...
Total records in 5k dataset: 64,877
Partitions: train=45,302 | val=9,730 | test=9,845
Active datasets: train=45,302 | val=400 | test=1,011 (intact test abstracts: 74)


In [5]:
# ==============================================================================
# STEP 5: 4-BIT NF4 QUANTIZED MODEL & LORA INITIALIZATION
# ==============================================================================
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print("=" * 80)
print(f"INITIALIZING BASE MODEL: {SHORT_NAME} ({MODEL_ID})")
print(f"Target GPU: {GPU_NAME} | Compute Dtype: {COMPUTE_DTYPE}")
print("=" * 80)

tok_kwargs = {"token": hf_token} if hf_token else {}

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True, **tok_kwargs)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Ensure tokenizer has a valid chat template
if getattr(tokenizer, "chat_template", None) is None:
    print("Tokenizer has no chat template -> installing standard Mistral-style [INST] template.")
    tokenizer.chat_template = (
        "{{ bos_token }}{% for m in messages %}"
        "{% if m['role'] == 'user' %}{{ '[INST] ' + m['content'] + ' [/INST]' }}"
        "{% elif m['role'] == 'assistant' %}{{ m['content'] + eos_token }}{% endif %}"
        "{% endfor %}"
    )

def load_base_model():
    last_err = None
    for attn in ("sdpa", None):
        for dtype_key in ("dtype", "torch_dtype"):
            kwargs = dict(quantization_config=bnb_config, device_map={"": 0}, **tok_kwargs)
            kwargs[dtype_key] = COMPUTE_DTYPE
            if attn:
                kwargs["attn_implementation"] = attn
            try:
                return AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs)
            except TypeError as e:
                last_err = e
            except ValueError as e:
                if attn:
                    last_err = e
                    break
                raise
    raise last_err

torch.cuda.empty_cache()
print("Loading 4-bit base model directly onto GPU 0 ...")
model = load_base_model()

model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.config.use_cache = False

peft_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES, bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
print("\nLoRA adapter attached successfully:")
model.print_trainable_parameters()
print(f"VRAM Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB / Reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB (of {VRAM_GB:.2f} GB)")

INITIALIZING BASE MODEL: BioMistral-7B (BioMistral/BioMistral-7B)
Target GPU: Tesla T4 | Compute Dtype: torch.float16


config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

...

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 14.5GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]


LoRA adapter attached successfully:
trainable params: 41,943,040 || all params: 7,283,675,136 || trainable%: 0.5758
VRAM Allocated: 4.49 GB / Reserved: 4.52 GB (of 14.56 GB)


In [6]:
# ==============================================================================
# STEP 6: COMPLETION-ONLY LOSS TOKENIZATION & AUDIT
# ==============================================================================
MERGE_SYSTEM = True

def format_messages_for_model(messages):
    if not messages:
        return []
    if MERGE_SYSTEM and messages[0].get("role") == "system":
        sys_c = messages[0].get("content", "")
        usr_c = messages[1].get("content", "") if len(messages) > 1 else ""
        merged = [{"role": "user", "content": f"{sys_c}\n\n{usr_c}".strip()}]
        return merged + list(messages[2:])
    return list(messages)

HEAD_KEEP = 8

def _truncate_middle(ids, budget):
    if len(ids) <= budget:
        return ids, False
    head = min(HEAD_KEEP, budget // 4)
    return ids[:head] + ids[len(ids) - (budget - head):], True

def build_prompt_ids(messages, tokenizer, max_prompt_len):
    msgs = format_messages_for_model(messages)
    if msgs and msgs[-1].get("role") == "assistant":
        msgs = msgs[:-1]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    return _truncate_middle(ids, max_prompt_len)

def build_example(messages, tokenizer, max_length):
    label = format_messages_for_model(messages)[-1]["content"].strip()
    answer_ids = tokenizer(label, add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]
    prompt_ids, truncated = build_prompt_ids(messages, tokenizer, max_length - len(answer_ids))
    return {
        "input_ids": prompt_ids + answer_ids,
        "attention_mask": [1] * (len(prompt_ids) + len(answer_ids)),
        "labels": [-100] * len(prompt_ids) + answer_ids,
        "truncated": truncated,
    }

def data_collator(features):
    max_len = max(len(f["input_ids"]) for f in features)
    pad_id = tokenizer.pad_token_id
    ids, mask, labels = [], [], []
    for f in features:
        pad = max_len - len(f["input_ids"])
        ids.append(list(f["input_ids"]) + [pad_id] * pad)
        mask.append(list(f["attention_mask"]) + [0] * pad)
        labels.append(list(f["labels"]) + [-100] * pad)
    return {"input_ids": torch.tensor(ids), "attention_mask": torch.tensor(mask), "labels": torch.tensor(labels)}

# Audit check on sample 0
ex = build_example(ds_train[0]["messages"], tokenizer, MAX_SEQ_LENGTH)
n_prompt = sum(1 for x in ex["labels"] if x == -100)
print("=" * 80)
print("COMPLETION-ONLY LOSS AUDIT")
print("=" * 80)
print(f"Sequence length: {len(ex['input_ids'])} | Masked prompt tokens: {n_prompt} | Supervised tokens: {len(ex['input_ids']) - n_prompt}")
assert (len(ex["input_ids"]) - n_prompt) > 0, "No supervised label tokens found!"
print("[PASS] Loss is strictly computed on assistant discourse label tokens.")

def preprocess_function(examples):
    out = {"input_ids": [], "attention_mask": [], "labels": []}
    for msgs in examples["messages"]:
        item = build_example(msgs, tokenizer, MAX_SEQ_LENGTH)
        out["input_ids"].append(item["input_ids"])
        out["attention_mask"].append(item["attention_mask"])
        out["labels"].append(item["labels"])
    return out

print("Tokenizing train and val datasets ...")
tokenized_train = ds_train.map(preprocess_function, batched=True, batch_size=256, remove_columns=ds_train.column_names, desc="train")
tokenized_val   = ds_val.map(preprocess_function, batched=True, batch_size=256, remove_columns=ds_val.column_names, desc="val")
print(f"Tokenized: train={len(tokenized_train):,} | val={len(tokenized_val):,}")

COMPLETION-ONLY LOSS AUDIT
Sequence length: 512 | Masked prompt tokens: 508 | Supervised tokens: 4
[PASS] Loss is strictly computed on assistant discourse label tokens.
Tokenizing train and val datasets ...


train:   0%|          | 0/45302 [00:00<?, ? examples/s]

val:   0%|          | 0/400 [00:00<?, ? examples/s]

Tokenized: train=45,302 | val=400


In [7]:
# ==============================================================================
# STEP 7: PRE-FLIGHT BENCHMARK & AUTOMATIC TIME-BUDGET FIT
# ==============================================================================
print("=" * 80)
print("PRE-FLIGHT COMPUTE BENCHMARK")
print("=" * 80)

bench_rng = random.Random(123)
bench_idx = bench_rng.sample(range(len(tokenized_train)), min(len(tokenized_train), PER_DEVICE_BATCH_SIZE * 8))
bench_batches = [data_collator([tokenized_train[i] for i in bench_idx[k:k + PER_DEVICE_BATCH_SIZE]])
                 for k in range(0, len(bench_idx) - PER_DEVICE_BATCH_SIZE + 1, PER_DEVICE_BATCH_SIZE)]

model.train()
torch.cuda.reset_peak_memory_stats()
WARMUP, times, losses = 2, [], []
for step, batch in enumerate(bench_batches):
    batch = {k: v.to("cuda:0") for k, v in batch.items()}
    torch.cuda.synchronize(); t0 = time.time()
    with torch.autocast("cuda", dtype=COMPUTE_DTYPE):
        loss = model(**batch).loss
    loss.backward()
    torch.cuda.synchronize(); dt = time.time() - t0
    model.zero_grad(set_to_none=True)
    losses.append(loss.item())
    if step >= WARMUP:
        times.append(dt)
    print(f"  micro-batch {step}: {dt:.2f}s | loss={loss.item():.3f} | seq_len={batch['input_ids'].shape[1]}")

sec_per_micro = float(np.mean(times)) if times else 2.5
sec_per_sample = sec_per_micro / PER_DEVICE_BATCH_SIZE
peak_gb = torch.cuda.max_memory_allocated() / 1024 ** 3
print(f"\nMeasured: {sec_per_micro:.2f} s / micro-batch ({PER_DEVICE_BATCH_SIZE} samples) -> {sec_per_sample:.3f} s / sample | peak VRAM {peak_gb:.2f} GB")

OVERHEAD = 1.10
n_total = len(tokenized_train)
proj_hours = n_total * NUM_TRAIN_EPOCHS * sec_per_sample * OVERHEAD / 3600
print(f"Projected time for {n_total:,} samples: {proj_hours:.2f} h (Budget: {MAX_TRAIN_HOURS} h)")

if proj_hours > MAX_TRAIN_HOURS:
    fit = int(MAX_TRAIN_HOURS * 3600 / (sec_per_sample * OVERHEAD) / NUM_TRAIN_EPOCHS)
    fit = (fit // EFFECTIVE_BATCH) * EFFECTIVE_BATCH
    if AUTO_FIT_TO_BUDGET and fit >= EFFECTIVE_BATCH * 4:
        tokenized_train = tokenized_train.select(range(fit))
        print(f"AUTO-FIT: Training set shrunken to {fit:,} samples to guarantee completion within {MAX_TRAIN_HOURS} h limit.")
    else:
        print("WARNING: Run projected to exceed budget; TimeBudgetCallback will safeguard completion.")
else:
    print("Full training set fits safely inside the session budget.")

del bench_batches
gc.collect(); torch.cuda.empty_cache()

PRE-FLIGHT COMPUTE BENCHMARK
  micro-batch 0: 5.39s | loss=1.061 | seq_len=360
  micro-batch 1: 6.14s | loss=1.206 | seq_len=512
  micro-batch 2: 5.33s | loss=1.235 | seq_len=428
  micro-batch 3: 4.48s | loss=1.457 | seq_len=359
  micro-batch 4: 5.08s | loss=0.952 | seq_len=396
  micro-batch 5: 5.76s | loss=1.609 | seq_len=455
  micro-batch 6: 4.58s | loss=1.097 | seq_len=372
  micro-batch 7: 5.12s | loss=1.793 | seq_len=388

Measured: 5.06 s / micro-batch (4 samples) -> 1.264 s / sample | peak VRAM 6.54 GB
Projected time for 45,302 samples: 17.50 h (Budget: 8.5 h)
AUTO-FIT: Training set shrunken to 22,000 samples to guarantee completion within 8.5 h limit.


In [8]:
# ==============================================================================
# STEP 8: QLORA TRAINING WITH RESUMABLE CHECKPOINTS
# ==============================================================================
from transformers import TrainingArguments, Trainer, TrainerCallback
from transformers.trainer_utils import get_last_checkpoint

class TimeBudgetCallback(TrainerCallback):
    def __init__(self, max_seconds):
        self.max_seconds, self.t0 = max_seconds, None
    def on_train_begin(self, args, state, control, **kw):
        self.t0 = time.time()
    def on_step_end(self, args, state, control, **kw):
        if self.t0 and time.time() - self.t0 > self.max_seconds:
            print(f"\n[TimeBudget] {self.max_seconds / 3600:.2f} h reached -> saving checkpoint and stopping.")
            control.should_save = True
            control.should_training_stop = True
        return control

ta_params = inspect.signature(TrainingArguments.__init__).parameters
ta_kwargs = dict(
    output_dir=checkpoint_dir,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    max_grad_norm=1.0,
    bf16=USE_BF16,
    fp16=(not USE_BF16),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim=OPTIMIZER_NAME,
    logging_steps=LOG_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    eval_steps=SAVE_STEPS,
    prediction_loss_only=True,
    remove_unused_columns=False,
    dataloader_num_workers=2,
    report_to="none",
    seed=RANDOM_SEED,
)
ta_kwargs["eval_strategy" if "eval_strategy" in ta_params else "evaluation_strategy"] = "steps"
if "warmup_ratio" in ta_params:
    ta_kwargs["warmup_ratio"] = WARMUP_RATIO

training_args = TrainingArguments(**{k: v for k, v in ta_kwargs.items() if k in ta_params})

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    callbacks=[TimeBudgetCallback(MAX_TRAIN_HOURS * 3600 * 1.05)],
)

steps_per_epoch = max(1, len(tokenized_train) // EFFECTIVE_BATCH)
print(f"Training on {len(tokenized_train):,} samples | Effective batch {EFFECTIVE_BATCH} | ~{steps_per_epoch * NUM_TRAIN_EPOCHS:,} steps")

last_ckpt = get_last_checkpoint(checkpoint_dir)
print(f"Resuming from {last_ckpt}" if last_ckpt else "No checkpoint found -> training from start.")

model.config.use_cache = False
t_train0 = time.time()
trainer.train(resume_from_checkpoint=last_ckpt)
train_hours = (time.time() - t_train0) / 3600

TRAIN_INFO = {
    "model_name": SHORT_NAME,
    "train_samples": len(tokenized_train),
    "epochs": NUM_TRAIN_EPOCHS,
    "effective_batch": EFFECTIVE_BATCH,
    "learning_rate": LEARNING_RATE,
    "max_seq_length": MAX_SEQ_LENGTH,
    "optimizer_steps": int(trainer.state.global_step),
    "train_hours": round(train_hours, 3),
}
print(f"\nTraining Complete: {TRAIN_INFO}")

# Save LoRA adapter & light fp16 copy
final_adapter_dir = os.path.join(output_run_dir, "final_adapter")
trainer.save_model(final_adapter_dir)
tokenizer.save_pretrained(final_adapter_dir)

from safetensors.torch import load_file, save_file
slim_dir = os.path.join(output_run_dir, "final_adapter_fp16")
os.makedirs(slim_dir, exist_ok=True)
safetensors_file = os.path.join(final_adapter_dir, "adapter_model.safetensors")
bin_file = os.path.join(final_adapter_dir, "adapter_model.bin")

if os.path.exists(safetensors_file):
    sd = load_file(safetensors_file)
    save_file({k: v.to(torch.float16).contiguous() for k, v in sd.items()}, os.path.join(slim_dir, "adapter_model.safetensors"), metadata={"format": "pt"})
elif os.path.exists(bin_file):
    sd = torch.load(bin_file, map_location="cpu")
    save_file({k: v.to(torch.float16).contiguous() for k, v in sd.items()}, os.path.join(slim_dir, "adapter_model.safetensors"), metadata={"format": "pt"})

if os.path.exists(os.path.join(final_adapter_dir, "adapter_config.json")):
    shutil.copy2(os.path.join(final_adapter_dir, "adapter_config.json"), os.path.join(slim_dir, "adapter_config.json"))

with open(os.path.join(output_run_dir, f"{SHORT_NAME}_training_log.json"), "w") as f:
    json.dump({"train_info": TRAIN_INFO, "log_history": trainer.state.log_history}, f, indent=2)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
tr = [(h["step"], h["loss"]) for h in trainer.state.log_history if "loss" in h]
ev = [(h["step"], h["eval_loss"]) for h in trainer.state.log_history if "eval_loss" in h]
plt.figure(figsize=(7, 4))
if tr: plt.plot(*zip(*tr), label="train loss")
if ev: plt.plot(*zip(*ev), "o-", label="val loss")
plt.xlabel("optimizer step"); plt.ylabel("loss"); plt.legend(); plt.title(f"{SHORT_NAME} Training Curve"); plt.tight_layout()
plt.savefig(os.path.join(output_run_dir, f"{SHORT_NAME}_loss_curve.png"), dpi=200)
plt.close()


Training on 22,000 samples | Effective batch 16 | ~1,375 steps
No checkpoint found -> training from start.


Step,Training Loss,Validation Loss
250,0.106554,0.111730
500,0.093278,0.096654
750,0.084952,0.114788
1000,0.057282,0.088246
1250,0.064682,0.077601
1375,0.046066,0.077737



Training Complete: {'model_name': 'BioMistral-7B', 'train_samples': 22000, 'epochs': 1, 'effective_batch': 16, 'learning_rate': 0.0002, 'max_seq_length': 512, 'optimizer_steps': 1375, 'train_hours': 8.714}


In [9]:
# ==============================================================================
# STEP 9: TEST-SET INFERENCE & PROBABILITY EXTRACTION (FOR ENSEMBLING)
# ==============================================================================
print("=" * 80)
print(f"GENERATION & PROBABILITY EXTRACTION ON TEST SET ({len(ds_test):,} sentences)")
print("=" * 80)

model.eval()
# Disable gradient checkpointing so generation can safely use KV cache
if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()
if hasattr(getattr(model, "base_model", None), "gradient_checkpointing_disable"):
    model.base_model.gradient_checkpointing_disable()
model.config.use_cache = True
torch.cuda.empty_cache()

def parse_llm_output(raw_text: str):
    cleaned = re.sub(r"[^A-Za-z ]", " ", raw_text).strip().upper()
    if cleaned in LABEL2ID:
        return cleaned, "exact"
    m = re.search(r"(BACKGROUND|OBJECTIVE|METHODS|RESULTS|CONCLUSION)", cleaned)
    if m:
        return m.group(1), "lenient"
    return None, "unparseable"

test_msgs = [r["messages"] for r in ds_test]
true_labels = [get_label(r) for r in test_records]
prompt_ids_all = [build_prompt_ids(m, tokenizer, MAX_SEQ_LENGTH - 8)[0] for m in test_msgs]
order = sorted(range(len(prompt_ids_all)), key=lambda i: len(prompt_ids_all[i]))

raw_outputs = [None] * len(prompt_ids_all)
probs_all = np.zeros((len(prompt_ids_all), NUM_CLASSES), dtype=np.float32)
pad_id = tokenizer.pad_token_id
t_gen0 = time.time()

for b, start in enumerate(range(0, len(order), EVAL_BATCH_SIZE)):
    idxs = order[start:start + EVAL_BATCH_SIZE]
    max_len = max(len(prompt_ids_all[i]) for i in idxs)
    ids = torch.full((len(idxs), max_len), pad_id, dtype=torch.long)
    att = torch.zeros((len(idxs), max_len), dtype=torch.long)
    for r_, i in enumerate(idxs):
        p = prompt_ids_all[i]
        ids[r_, max_len - len(p):] = torch.tensor(p)
        att[r_, max_len - len(p):] = 1
        
    with torch.no_grad(), torch.autocast("cuda", dtype=COMPUTE_DTYPE):
        out = model.generate(
            input_ids=ids.to("cuda:0"), attention_mask=att.to("cuda:0"),
            max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
            pad_token_id=pad_id, use_cache=True,
            return_dict_in_generate=True
        )
    
    gen_seqs = out.sequences
    for r_, i in enumerate(idxs):
        raw_outputs[i] = tokenizer.decode(gen_seqs[r_, max_len:], skip_special_tokens=True)
        p_label, kind = parse_llm_output(raw_outputs[i])
        if p_label and p_label in LABEL2ID:
            c_idx = LABEL2ID[p_label]
            probs_all[i, :] = 0.05 / (NUM_CLASSES - 1)
            probs_all[i, c_idx] = 0.95
        else:
            probs_all[i, :] = 1.0 / NUM_CLASSES
            
    if b % 25 == 0:
        print(f"  Processed {min(start + EVAL_BATCH_SIZE, len(order)):5d}/{len(order)} | {time.time() - t_gen0:.0f}s")

parsed = [parse_llm_output(t) for t in raw_outputs]
generated_predictions = [p[0] for p in parsed]
parse_kind = [p[1] for p in parsed]

prob_save_path = os.path.join(output_run_dir, f"{SHORT_NAME}_test_probabilities.npy")
np.save(prob_save_path, probs_all)
np.save(os.path.join(WORKING_DIR, f"{SHORT_NAME}_test_probabilities.npy"), probs_all)


GENERATION & PROBABILITY EXTRACTION ON TEST SET (1,011 sentences)
  Processed     8/1011 | 3s
  Processed   208/1011 | 77s
  Processed   408/1011 | 158s
  Processed   608/1011 | 242s
  Processed   808/1011 | 330s
  Processed  1008/1011 | 432s


In [10]:
# ==============================================================================
# STEP 10: METRICS, CONFUSION MATRIX & TRANSITION CONSISTENCY
# ==============================================================================
import sklearn.metrics as metrics
import matplotlib.pyplot as plt
import seaborn as sns

y_true = np.array([LABEL2ID[l] for l in true_labels])
y_pred = np.array([LABEL2ID[p] if p is not None else UNPARSEABLE_ID for p in generated_predictions])
labels5 = list(range(NUM_CLASSES))

acc = metrics.accuracy_score(y_true, y_pred)
macro_f1 = metrics.f1_score(y_true, y_pred, labels=labels5, average="macro", zero_division=0)
micro_f1 = metrics.f1_score(y_true, y_pred, labels=labels5, average="micro", zero_division=0)
p_per, r_per, f1_per, support = metrics.precision_recall_fscore_support(y_true, y_pred, labels=labels5, zero_division=0)

paper_ids = [r.get("paper_id") for r in test_records]
df_eval = pd.DataFrame({
    "paper_id": paper_ids, "true_id": y_true, "pred_id": y_pred,
    "true": true_labels, "pred": [p if p else "UNPARSEABLE" for p in generated_predictions],
    "raw_output": raw_outputs
})

total_tr = legal_tr = 0
P_counts = np.zeros((NUM_CLASSES, NUM_CLASSES))
for _, grp in df_eval.groupby("paper_id", sort=False):
    seq = grp["pred_id"].tolist()
    for a, b in zip(seq[:-1], seq[1:]):
        if a < NUM_CLASSES and b < NUM_CLASSES:
            total_tr += 1
            P_counts[a, b] += 1
            if b >= a:
                legal_tr += 1

trans_rate = (legal_tr / total_tr * 100) if total_tr > 0 else 0.0
P_pred = np.divide(P_counts, P_counts.sum(1, keepdims=True), out=np.zeros_like(P_counts), where=P_counts.sum(1, keepdims=True) > 0)

cm = metrics.confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES + 1)))[:NUM_CLASSES]
col_names = TARGET_CLASSES + ["UNPARSEABLE"]
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", xticklabels=col_names, yticklabels=TARGET_CLASSES)
plt.title(f"{SHORT_NAME}\nMacro-F1 {macro_f1 * 100:.2f}% | Accuracy {acc * 100:.2f}%")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.tight_layout()
cm_path = os.path.join(output_run_dir, f"{SHORT_NAME}_confusion_matrix.png")
plt.savefig(cm_path, dpi=200); plt.close()

pd.DataFrame(cm, index=TARGET_CLASSES, columns=col_names).to_csv(os.path.join(output_run_dir, f"{SHORT_NAME}_confusion_matrix.csv"))
pd.DataFrame(P_pred, index=TARGET_CLASSES, columns=TARGET_CLASSES).to_csv(os.path.join(output_run_dir, f"{SHORT_NAME}_transition_matrix_pred.csv"))
df_eval.to_csv(os.path.join(output_run_dir, f"{SHORT_NAME}_test_predictions.csv"), index=False)
df_eval.to_csv(os.path.join(WORKING_DIR, f"{SHORT_NAME}_test_predictions.csv"), index=False)

single_metrics = {
    "model_name": SHORT_NAME, "base_model_id": MODEL_ID,
    "accuracy": float(acc), "macro_f1": float(macro_f1), "micro_f1": float(micro_f1),
    "transition_consistency_pct": float(trans_rate),
    "per_class": {TARGET_CLASSES[i]: {"precision": float(p_per[i]), "recall": float(r_per[i]), "f1": float(f1_per[i]), "support": int(support[i])} for i in range(NUM_CLASSES)}
}
with open(os.path.join(output_run_dir, f"{SHORT_NAME}_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(single_metrics, f, indent=2)

print("=" * 80)
print(f"RESULTS FOR {SHORT_NAME} (Test Set: {len(y_true)} sentences):")
print("=" * 80)
print(f"  Accuracy:               {acc * 100:.2f}%")
print(f"  Macro-F1:               {macro_f1 * 100:.2f}%")
print(f"  Micro-F1:               {micro_f1 * 100:.2f}%")
print(f"  Transition Legality:    {trans_rate:.2f}%")
for i, c in enumerate(TARGET_CLASSES):
    print(f"    {c:11s} P={p_per[i]*100:5.2f}%  R={r_per[i]*100:5.2f}%  F1={f1_per[i]*100:5.2f}%  (n={support[i]})")

RESULTS FOR BioMistral-7B (Test Set: 1011 sentences):
  Accuracy:               88.82%
  Macro-F1:               88.63%
  Micro-F1:               88.82%
  Transition Legality:    97.76%
    BACKGROUND  P=92.43%  R=94.48%  F1=93.44%  (n=181)
    OBJECTIVE   P=91.43%  R=76.19%  F1=83.12%  (n=126)
    METHODS     P=83.79%  R=88.33%  F1=86.00%  (n=240)
    RESULTS     P=90.77%  R=88.06%  F1=89.39%  (n=335)
    CONCLUSION  P=86.71%  R=96.12%  F1=91.18%  (n=129)


In [11]:
# ==============================================================================
# STEP 11: MODEL CARD & LATEX SNIPPET GENERATOR
# ==============================================================================
card_path = os.path.join(output_run_dir, "README_MODEL_CARD.md")
with open(card_path, "w", encoding="utf-8") as f:
    f.write(f"# Model Card: {SHORT_NAME} Rhetorical Role Classifier\n\n")
    f.write(f"- **Base Model:** `{MODEL_ID}`\n")
    f.write(f"- **Method:** QLoRA 4-bit NF4, compute dtype `{str(COMPUTE_DTYPE).replace('torch.', '')}`\n")
    f.write(f"- **Hardware:** {GPU_NAME}\n\n")
    f.write(f"| Metric | Value |\n| :-- | :-: |\n| Accuracy | {acc * 100:.2f}% |\n| Macro-F1 | {macro_f1 * 100:.2f}% |\n| Transition Legality | {trans_rate:.2f}% |\n\n")
    f.write("## Per-Class Results\n| Role | Precision | Recall | F1 | Support |\n| :-- | :-: | :-: | :-: | :-: |\n")
    for c in TARGET_CLASSES:
        d = single_metrics["per_class"][c]
        f.write(f"| **{c}** | {d['precision'] * 100:.2f}% | {d['recall'] * 100:.2f}% | {d['f1'] * 100:.2f}% | {d['support']} |\n")
print(f"Saved model card: {card_path}")

Saved model card: /kaggle/working/models/llm_qlora/BioMistral-7B/README_MODEL_CARD.md
